# Macro Event Calendar

In [1]:
import sys, datetime as dt, os
sys.path.insert(0, "..")  # so `import macro_calendar` finds the sibling package

import polars as pl
import plotly.graph_objects as go

from macro_calendar.pipeline import build_calendar, filter_window, tier, around
from macro_calendar import cache

df = build_calendar(refresh=True)
df.shape

fred.fetch skipped: FRED_API_KEY not set


Retrying macro_calendar._http.get in 1 seconds as it raised HTTPStatusError: Client error '404 Not Found' for url 'https://www.ecb.europa.eu/press/calendars/mgcgc/shared/calendar.ics'
For more information check: https://developer.mozilla.org/en-US/docs/Web/HTTP/Status/404.


Retrying macro_calendar._http.get in 2 seconds as it raised HTTPStatusError: Client error '404 Not Found' for url 'https://www.ecb.europa.eu/press/calendars/mgcgc/shared/calendar.ics'
For more information check: https://developer.mozilla.org/en-US/docs/Web/HTTP/Status/404.


Retrying macro_calendar._http.get in 4 seconds as it raised HTTPStatusError: Client error '404 Not Found' for url 'https://www.ecb.europa.eu/press/calendars/mgcgc/shared/calendar.ics'
For more information check: https://developer.mozilla.org/en-US/docs/Web/HTTP/Status/404.


Retrying macro_calendar._http.get in 8 seconds as it raised HTTPStatusError: Client error '404 Not Found' for url 'https://www.ecb.europa.eu/press/calendars/mgcgc/shared/calendar.ics'
For more information check: https://developer.mozilla.org/en-US/docs/Web/HTTP/Status/404.


...

Retrying macro_calendar._http.get in 2 seconds as it raised HTTPStatusError: Client error '404 Not Found' for url 'https://www.ecb.europa.eu/press/calendars/mgcgc/html/index.en.ics'
For more information check: https://developer.mozilla.org/en-US/docs/Web/HTTP/Status/404.


Retrying macro_calendar._http.get in 4 seconds as it raised HTTPStatusError: Client error '404 Not Found' for url 'https://www.ecb.europa.eu/press/calendars/mgcgc/html/index.en.ics'
For more information check: https://developer.mozilla.org/en-US/docs/Web/HTTP/Status/404.


Retrying macro_calendar._http.get in 8 seconds as it raised HTTPStatusError: Client error '404 Not Found' for url 'https://www.ecb.europa.eu/press/calendars/mgcgc/html/index.en.ics'
For more information check: https://developer.mozilla.org/en-US/docs/Web/HTTP/Status/404.


ecb: ICS unavailable, using hardcoded date table


(4483, 12)

In [2]:
print("by source:")
print(df.group_by("source").len().sort("len", descending=True))
print()
print("by event_name (top 15):")
print(df.group_by("event_name").len().sort("len", descending=True).head(15))

by source:
shape: (4, 2)
┌──────────────┬──────┐
│ source       ┆ len  │
│ ---          ┆ ---  │
│ str          ┆ u32  │
╞══════════════╪══════╡
│ treasury_api ┆ 2662 │
│ computed     ┆ 1163 │
│ fed_board    ┆ 578  │
│ ecb_ics      ┆ 80   │
└──────────────┴──────┘

by event_name (top 15):
shape: (15, 2)
┌─────────────────────────────────┬─────┐
│ event_name                      ┆ len │
│ ---                             ┆ --- │
│ str                             ┆ u32 │
╞═════════════════════════════════╪═════╡
│ 2-Year Note Auction             ┆ 690 │
│ 5-Year Note Auction             ┆ 436 │
│ 10-Year Note Auction            ┆ 357 │
│ ISM Services PMI                ┆ 341 │
│ ISM Manufacturing PMI           ┆ 341 │
│ …                               ┆ …   │
│ 7-Year Note Auction             ┆ 267 │
│ Treasury Refunding: Financing … ┆ 116 │
│ Treasury Refunding: Details     ┆ 116 │
│ Quad Witching                   ┆ 113 │
│ 20-Year Bond Auction            ┆ 93  │
└──────────────────────

In [3]:
today = dt.date.today()
recent = filter_window(tier(df, 2), today - dt.timedelta(days=730), today + dt.timedelta(days=60))

fig = go.Figure()
for cat in sorted(recent["event_category"].unique()):
    sub = recent.filter(pl.col("event_category") == cat)
    fig.add_trace(go.Scatter(
        x=sub["release_date"],
        y=sub["tier"],
        mode="markers",
        name=cat,
        marker=dict(size=8, opacity=0.7),
        text=[f"{n} | {p or ''}" for n, p in zip(sub["event_name"], sub["period"])],
        hovertemplate="%{x|%Y-%m-%d}<br>%{text}<extra>" + cat + "</extra>",
    ))
fig.update_layout(
    title="Macro events timeline (Tier <= 2, last 2 years + 60d ahead)",
    yaxis=dict(title="tier", autorange="reversed", tickvals=[1, 2]),
    template="plotly_dark",
    height=420,
)
fig.show()

In [4]:
upcoming = (
    filter_window(tier(df, 1), today, today + dt.timedelta(days=30))
      .select(["release_date", "release_time_et", "event_name", "period", "source"])
      .sort(["release_date", "release_time_et"], nulls_last=True)
)
print(upcoming)

shape: (2, 5)
┌──────────────┬─────────────────┬───────────────┬─────────────────┬───────────┐
│ release_date ┆ release_time_et ┆ event_name    ┆ period          ┆ source    │
│ ---          ┆ ---             ┆ ---           ┆ ---             ┆ ---       │
│ date         ┆ time            ┆ str           ┆ str             ┆ str       │
╞══════════════╪═════════════════╪═══════════════╪═════════════════╪═══════════╡
│ 2026-05-20   ┆ 14:00:00        ┆ FOMC Minutes  ┆ FOMC 2026-04-29 ┆ fed_board │
│ 2026-06-17   ┆ 14:00:00        ┆ FOMC Decision ┆ null            ┆ fed_board │
└──────────────┴─────────────────┴───────────────┴─────────────────┴───────────┘


In [5]:
import numpy as np
import pandas as pd
import pandas_datareader.data as web

# Recompute rMOVE_rolling inline (same construction as notebooks/realized_move.ipynb).
START_RM = "2020-01-01"
WINDOW = 21
WEIGHTS = {"y2": 0.20, "y5": 0.20, "y10": 0.40, "y30": 0.20}
FRED_TENORS = {"y2": "DGS2", "y5": "DGS5", "y10": "DGS10", "y30": "DGS30"}

raw = web.DataReader(list(FRED_TENORS.values()), "fred", START_RM)
raw.columns = list(FRED_TENORS)
raw = raw.dropna()
rmove = sum(
    w * np.sqrt((raw[t].diff() * 100).pow(2).rolling(WINDOW).mean()) * np.sqrt(252)
    for t, w in WEIGHTS.items()
)
rmove = rmove.dropna()

# FOMC + CPI markers from the calendar
events = df.filter(
    pl.col("event_name").is_in(["FOMC Decision", "CPI"])
).filter(
    (pl.col("release_date") >= dt.date.fromisoformat(START_RM)) &
    (pl.col("release_date") <= dt.date.today())
).to_pandas()

fig = go.Figure()
fig.add_trace(go.Scatter(x=rmove.index, y=rmove.values, mode="lines",
                        name="rMOVE rolling 21d", line=dict(color="#88c")))
for _, ev in events.iterrows():
    color = "#e66" if ev["event_name"] == "FOMC Decision" else "#6c6"
    fig.add_vline(x=ev["release_date"], line_color=color, line_width=0.5, opacity=0.4)
fig.add_trace(go.Scatter(x=[None], y=[None], mode="lines",
                        line=dict(color="#e66"), name="FOMC Decision"))
fig.add_trace(go.Scatter(x=[None], y=[None], mode="lines",
                        line=dict(color="#6c6"), name="CPI"))
fig.update_layout(
    title="rMOVE (10y-only) with FOMC and CPI markers",
    yaxis_title="bp/year",
    template="plotly_dark",
    yaxis=dict(showgrid=False),
    height=460,
)
fig.show()
print(f"rMOVE samples: {len(rmove):,}, events overlaid: {len(events):,}")

rMOVE samples: 1,573, events overlaid: 50


In [6]:
# Validation table per spec
def per_year_count(name: str) -> pl.DataFrame:
    return (
        df.filter(pl.col("event_name") == name)
          .with_columns(pl.col("release_date").dt.year().alias("year"))
          .group_by("year").len()
          .sort("year")
    )

print("FOMC Decisions per year (should be 8 from 1994-present):")
print(per_year_count("FOMC Decision"))
print()
print("CPI per year (should be 12 when FRED_API_KEY is set):")
print(per_year_count("CPI"))
print()
print("NFP per year (Employment Situation, 12/yr expected):")
print(per_year_count("Employment Situation (NFP)"))
print()
print("10y Note Auction per year (~12 expected):")
print(per_year_count("10-Year Note Auction"))
print()
print("Source range and null release_time_et counts:")
print(
    df.group_by("source").agg([
        pl.col("release_date").min().alias("earliest"),
        pl.col("release_date").max().alias("latest"),
        pl.col("release_time_et").is_null().sum().alias("null_times"),
        pl.len().alias("rows"),
    ]).sort("source")
)
print()
print("Pulls audit:")
print(cache.pulls_table())

FOMC Decisions per year (should be 8 from 1994-present):
shape: (34, 2)
┌──────┬─────┐
│ year ┆ len │
│ ---  ┆ --- │
│ i32  ┆ u32 │
╞══════╪═════╡
│ 1994 ┆ 8   │
│ 1995 ┆ 8   │
│ 1996 ┆ 8   │
│ 1997 ┆ 8   │
│ 1998 ┆ 8   │
│ …    ┆ …   │
│ 2023 ┆ 8   │
│ 2024 ┆ 8   │
│ 2025 ┆ 8   │
│ 2026 ┆ 8   │
│ 2027 ┆ 8   │
└──────┴─────┘

CPI per year (should be 12 when FRED_API_KEY is set):
shape: (0, 2)
┌──────┬─────┐
│ year ┆ len │
│ ---  ┆ --- │
│ i32  ┆ u32 │
╞══════╪═════╡
└──────┴─────┘

NFP per year (Employment Situation, 12/yr expected):
shape: (0, 2)
┌──────┬─────┐
│ year ┆ len │
│ ---  ┆ --- │
│ i32  ┆ u32 │
╞══════╪═════╡
└──────┴─────┘

10y Note Auction per year (~12 expected):
shape: (48, 2)
┌──────┬─────┐
│ year ┆ len │
│ ---  ┆ --- │
│ i32  ┆ u32 │
╞══════╪═════╡
│ 1979 ┆ 1   │
│ 1980 ┆ 3   │
│ 1981 ┆ 4   │
│ 1982 ┆ 4   │
│ 1983 ┆ 4   │
│ …    ┆ …   │
│ 2022 ┆ 12  │
│ 2023 ┆ 12  │
│ 2024 ┆ 12  │
│ 2025 ┆ 12  │
│ 2026 ┆ 5   │
└──────┴─────┘

Source range and null release_time_et coun